In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.6 MB/s eta 0:00:00


In [42]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import StackingRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import StackingRegressor, ExtraTreesRegressor
from sklearn.linear_model import RidgeCV, ElasticNetCV
from sklearn.svm import SVR


from sklearn.model_selection import RandomizedSearchCV
import numpy as np

from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.model_selection import cross_val_predict


from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

In [28]:
train = pd.read_csv("/content/drive/MyDrive/dsn26/processed_train.csv")
test = pd.read_csv("/content/drive/MyDrive/dsn26/processed_test.csv")
y = pd.read_csv("/content/drive/MyDrive/dsn26/target.csv")

In [29]:
y = y.total_sales

In [30]:
n_train = train.shape[0]
n_train

6818

In [31]:
display(train.head())
print("\n\n")
display(test.head())

,id,product_weight_kg,fat_content,shelf_visibility,product_price,store_age_years,store_size,store_location_tier,store_format,price_vs_cat,price_rank_in_cat,price_vs_store,price_per_kg,weight_vs_cat,vis_vs_product,vis_x_price
0,row_00000,14.252000,20,0.0271,81.37,45,30,30,20,0.630238,0.234813,0.575641,5.709374,1.115616,0.409863,2.205127
1,row_00001,7.698000,20,0.0720,42.05,35,10,10,20,0.326007,0.086538,0.295775,5.462458,0.638414,1.088934,3.027600
2,row_00002,14.264000,10,0.0421,41.35,33,20,10,20,0.298018,0.023112,0.294545,2.898906,1.155650,0.636724,1.740835
3,row_00003,12.899667,10,0.0449,174.35,47,20,30,40,1.214306,0.732584,1.247467,13.515853,1.091619,0.679071,7.828315
4,row_00004,10.338000,10,0.0120,203.06,28,10,20,20,1.384185,0.868235,1.419477,19.642097,0.838375,0.181489,2.436720


,id,product_weight_kg,fat_content,shelf_visibility,product_price,store_age_years,store_size,store_location_tier,store_format,price_vs_cat,price_rank_in_cat,price_vs_store,price_per_kg,weight_vs_cat,vis_vs_product,vis_x_price
0,row_00009,8.628000,20,0.0167,190.72,23,20,30,30,1.244705,0.748352,1.345452,22.104775,0.643209,0.252572,3.185024
1,row_00015,6.993000,20,0.0579,261.07,28,10,20,20,2.024034,0.982692,1.824992,37.333047,0.579947,0.875684,15.115953
2,row_00019,17.167667,20,0.1262,112.50,47,20,30,40,0.782337,0.328328,0.804933,6.553016,1.307324,1.908659,14.197500
3,row_00020,14.034000,10,0.0761,200.71,30,10,20,20,1.554566,0.802570,1.424928,14.301696,1.098552,1.150943,15.274031
4,row_00023,13.063000,20,0.0650,150.59,33,20,10,20,1.166370,0.586449,1.072687,11.527980,1.022544,0.983065,9.788350


In [32]:
test_id = test['id']

In [33]:
train.drop("id", axis=1, inplace=True, errors = "ignore")
test.drop("id", axis=1, inplace=True, errors = "ignore")

- Using best performing columns from catboost - `store_format`,	`price_vs_store` and `product_price`

In [34]:
train = train[["store_format",	"price_vs_store",	"product_price"]]
test = test[["store_format",	"price_vs_store",	"product_price"]]

In [35]:
categorical_cols = ["store_format"]

## Stacking

In [36]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

base_learners = [
    ("xgb", XGBRegressor(
        n_estimators=300, learning_rate=0.03, max_depth=3,
        subsample=0.9, colsample_bytree=0.8, min_child_weight=3,
        random_state=42, n_jobs=4)),
    ("cat", CatBoostRegressor(
        iterations=2000, learning_rate=0.02, depth=4, l2_leaf_reg=3,
        subsample=0.8, bootstrap_type="Bernoulli", rsm=0.8,
        random_seed=42, verbose=0, thread_count=4)),
    ("et", ExtraTreesRegressor(
        n_estimators=500, min_samples_leaf=2, max_features=0.7,
        random_state=42, n_jobs=4)),
    ("enet", make_pipeline(StandardScaler(),
        ElasticNetCV(l1_ratio=[.1, .5, .9, 1], cv=kf, n_jobs=4))),
    ("svr", make_pipeline(StandardScaler(), SVR(C=3, epsilon=0.05))),
]

stacking_reg = StackingRegressor(
    estimators=base_learners,
    final_estimator=RidgeCV(alphas=np.logspace(-3, 3, 13)),
    cv=kf,
    passthrough=False,
    n_jobs=1,
)

stacking_reg.fit(train,y)

StackingRegressor(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                  estimators=[('xgb',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.8, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=True,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_poli...
                                                             n_jobs=4))])),
                              ('svr',
                               Pipeline(steps=[('standardscaler',
                                                StandardScaler()),
                                               ('svr',
                                                SVR(C=3, epsilon=0.05))]))],
                  final_estimator=RidgeCV(alphas=array([1.00000000e-03, 3.16227766e-03, 1.00000000e-02, 3.16227766e-02,
       1.00000000e-01, 3.16227766e-01, 1.00000000e+00, 3.16227766e+00,
       1.00000000e+01, 3.16227766e+01, 1.00000000e+02, 3.16227766e+02,
       1.00000000e+03])),
                  n_jobs=1)

In [37]:
stacking_reg.score(train, y)

0.6106524165840354

In [38]:
from sklearn.model_selection import cross_val_score

for name, est in base_learners:
    s = cross_val_score(est, train, y, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=1)
    print(f"{name:6s} RMSE {-s.mean():.4f} ± {s.std():.4f}")

s = cross_val_score(stacking_reg, train, y, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=1)
print(f"STACK  RMSE {-s.mean():.4f} ± {s.std():.4f}")

xgb    RMSE 1084.8449 ± 18.6296
cat    RMSE 1098.2493 ± 17.0849
et     RMSE 1120.1416 ± 20.6950
enet   RMSE 1222.1434 ± 10.8673
svr    RMSE 1383.3394 ± 32.2228
STACK  RMSE 1084.4912 ± 19.1210


In [39]:
stacking_reg.final_estimator_.coef_

array([ 1.33989107, -0.37465382,  0.02289777, -0.00292438,  0.04937684])

In [40]:
import pandas as pd
from sklearn.metrics import root_mean_squared_error, r2_score

names = [name for name, _ in stacking_reg.estimators]
fitted = stacking_reg.named_estimators_

summary = pd.DataFrame({
    "model": names,
    "meta_weight": stacking_reg.final_estimator_.coef_,

}).sort_values(by = "meta_weight")

print(f"Meta intercept: {stacking_reg.final_estimator_.intercept_:.4f}")
print(f"Ridge alpha:    {stacking_reg.final_estimator_.alpha_}")
summary.round(4)

Meta intercept: -58.3244
Ridge alpha:    1000.0


,model,meta_weight
1,cat,-0.3747
3,enet,-0.0029
2,et,0.0229
4,svr,0.0494
0,xgb,1.3399


## Boosting

In [43]:
class ResidualBoostingRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, estimators, learning_rate=1.0, cv=None):
        self.estimators = estimators
        self.learning_rate = learning_rate
        self.cv = cv

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        self.init_ = y.mean()
        residual = y - self.init_
        self.estimators_ = []
        for name, est in self.estimators:
            # Out-of-fold predictions, so overfit models don't zero out the residuals
            oof = cross_val_predict(clone(est), X, residual, cv=self.cv)
            fitted = clone(est).fit(X, residual)
            self.estimators_.append((name, fitted))
            residual = residual - self.learning_rate * oof
        return self

    def predict(self, X):
        pred = np.full(len(X), self.init_)
        for _, est in self.estimators_:
            pred += self.learning_rate * est.predict(X)
        return pred


boost_order = [
    ("enet", make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[.1, .5, .9, 1], cv=kf, n_jobs=4))),
    ("xgb",  base_learners[0][1]),
    ("cat",  base_learners[1][1]),
    ("et",   base_learners[2][1]),
]

In [44]:
boost_reg = ResidualBoostingRegressor(boost_order, learning_rate=0.5, cv=kf)
boost_reg.fit(train, y)

ResidualBoostingRegressor(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                          estimators=[('enet',
                                       Pipeline(steps=[('standardscaler',
                                                        StandardScaler()),
                                                       ('elasticnetcv',
                                                        ElasticNetCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                                                                     l1_ratio=[0.1,
                                                                               0.5,
                                                                               0.9,
                                                                               1],
                                                                     n_jobs=4))])),
                                      ('xgb',
                                       XGBRegressor(base_score=None,
                                                    booster=None,
                                                    callbacks=None,
                                                    colsample_bylevel=No...
                                                    n_estimators=300, n_jobs=4,
                                                    num_parallel_tree=None, ...)),
                                      ('cat',
                                       CatBoostRegressor(bootstrap_type='Bernoulli', depth=4, iterations=2000, l2_leaf_reg=3, learning_rate=0.02, loss_function='RMSE', random_seed=42, rsm=0.8, subsample=0.8, thread_count=4, verbose=0)),
                                      ('et',
                                       ExtraTreesRegressor(max_features=0.7,
                                                           min_samples_leaf=2,
                                                           n_estimators=500,
                                                           n_jobs=4,
                                                           random_state=42))],
                          learning_rate=0.5)

In [45]:
boost_reg.score(train, y)

0.7016779898083485